# Chapter 8 — Text Data Analysis: Security Logs & Threat Intelligence

## 📋 Latihan Soal

### Dataset
- **Mirai Botnet Dataset** — https://huggingface.co/datasets/bornpresident/mirai_botnet
- Log message di-generate dari pattern traffic Mirai (2000 sample)
- 4 kelas: BenignTraffic, Mirai-greeth_flood, Mirai-udpplain, Mirai-greip_flood

### Teknik
Text cleaning, stop words, POS tagging, stemming, lemmatisation, ngrams, word clouds, TF-IDF, sentiment analysis, LDA topic modelling, coherence score


In [ ]:
# ============================================
# SETUP: Load Mirai Botnet Dataset & Generate Text Data
# ============================================
# Dataset: https://huggingface.co/datasets/bornpresident/mirai_botnet
# Text data di-generate dari pattern Mirai untuk keperluan analisis NLP
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load dataset dari HuggingFace
ds = load_dataset('bornpresident/mirai_botnet')
df = ds['train'].to_pandas()
df.columns = [c.replace(' ', '_') for c in df.columns]
df = df.rename(columns={'label': 'traffic_type'})

# Sample 2000 baris untuk performa
np.random.seed(42)
df = df.sample(n=2000, random_state=42).reset_index(drop=True)

# === Generate Text Data dari pattern Mirai ===
# Konversi setiap baris menjadi log message format
def generate_log(row):
    attack = row['traffic_type']
    proto = int(row.get('Protocol_Type', 6))
    rate = row.get('Rate', 0)
    duration = row.get('Duration', 0)
    if attack == 'Benign':
        msgs = [
            f"Normal traffic detected: protocol {proto}, rate {rate:.1f} pkts/s, duration {duration}s",
            f"Routine connection established, protocol type {proto}, flow duration {duration}s",
            f"Standard network activity observed, rate {rate:.1f}, protocol {proto}",
        ]
    elif 'greeth' in attack:
        msgs = [
            f"ALERT: GRE tunnel flooding detected, rate {rate:.1f} pkts/s, protocol {proto}",
            f"Mirai GRE flood attack: high rate {rate:.1f}, duration {duration}s, protocol {proto}",
            f"WARNING: GRE tunnel abuse detected, suspicious rate {rate:.1f} from botnet",
        ]
    elif 'udpplain' in attack:
        msgs = [
            f"ALERT: UDP flood detected, rate {rate:.1f} pkts/s, protocol {proto}",
            f"Mirai UDP plain attack: volume {rate:.1f}, duration {duration}s",
            f"WARNING: UDP amplification attack, rate {rate:.1f}, protocol {proto}",
        ]
    elif 'greip' in attack:
        msgs = [
            f"ALERT: GRE-IP flood detected, rate {rate:.1f} pkts/s, protocol {proto}",
            f"Mirai GRE-IP attack: rate {rate:.1f}, duration {duration}s, protocol {proto}",
            f"WARNING: GRE-IP tunnel flooding, suspicious rate {rate:.1f}",
        ]
    else:
        msgs = [f"Unknown traffic pattern detected: {attack}"]
    return msgs[int(row.name) % len(msgs)]

df['log_message'] = df.apply(generate_log, axis=1)

# Buat kolom untuk text analysis
df['attack_type'] = df['traffic_type'].apply(
    lambda x: 'Normal' if x == 'Benign' else x.replace('Mirai-', 'Mirai ')
)

print(f"Dataset shape: {df.shape}")
print(f"Traffic types:\n{df['traffic_type'].value_counts()}")
print(f"\nSample log messages:")
for _, row in df.head(3).iterrows():
    print(f"  [{row['traffic_type']}] {row['log_message'][:100]}...")


---
## Soal 1: Preparing Text Data

Bersihkan dan tokenize log message:
1. Convert ke lowercase
2. Hapus angka dan punctuation
3. Tokenize menjadi individual words
4. Buat kolom `cleaned_tokens`


In [ ]:

import re
import nltk
from nltk.tokenize import word_tokenize

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Clean and tokenize log messages
def clean_log(text):
    text = text.lower()
    text = re.sub(r'\d+\.\d+', 'NUM', text)  # Replace numbers
    text = re.sub(r'[^a-z\s]', '', text)  # Remove punctuation
    tokens = word_tokenize(text)
    return tokens

df['cleaned_tokens'] = df['log_message'].apply(clean_log)
print(f"Sample tokens:")
for _, row in df.head(3).iterrows():
    print(f"  [{row['traffic_type']}] {row['cleaned_tokens'][:10]}...")


---
## Soal 2: Removing Stop Words

Hapus stop words dari log messages. Tambahkan domain-specific stop words (detected, rate, protocol, dll). Kata apa yang paling sering muncul setelah filtering?


In [ ]:

from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))
stop_words.update(['detected', 'rate', 'protocol', 'duration', 'mirai', 'flood'])

df['filtered_tokens'] = df['cleaned_tokens'].apply(
    lambda tokens: [t for t in tokens if t not in stop_words]
)

# Most common words after filtering
all_tokens = [t for tokens in df['filtered_tokens'] for t in tokens]
from collections import Counter
top_words = Counter(all_tokens).most_common(15)
print("Top 15 kata setelah stop words dihapus:")
for word, count in top_words:
    print(f"  {word}: {count}")


---
## Soal 3: Analysing Part of Speech (POS)

Lakukan POS tagging pada semua log messages. Hitung distribusi POS tags. Temukan noun yang paling sering muncul.


In [ ]:

nltk.download('averaged_perceptron_tagger', quiet=True)

# POS tagging pada log messages
all_text = ' '.join(df['log_message'].values)
tokens = word_tokenize(all_text)
pos_tags = nltk.pos_tag(tokens)

# Distribusi POS tags
pos_counts = Counter(tag for _, tag in pos_tags)
print("=== Distribusi POS Tags ===")
for tag, count in pos_counts.most_common(10):
    print(f"  {tag}: {count}")

# Extract nouns
nouns = [word for word, tag in pos_tags if tag in ['NN', 'NNS']]
top_nouns = Counter(nouns).most_common(10)
print("\n=== Top 10 Nouns ===")
for word, count in top_nouns:
    print(f"  {word}: {count}")


---
## Soal 4: Stemming & Lemmatisation

Terapkan stemming (PorterStemmer) dan lemmatisation (WordNetLemmatizer). Bandingkan jumlah token unik. Mana yang lebih baik mempertahankan makna?


In [ ]:

from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet', quiet=True)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

df['stemmed'] = df['filtered_tokens'].apply(
    lambda tokens: [stemmer.stem(t) for t in tokens]
)
df['lemmatized'] = df['filtered_tokens'].apply(
    lambda tokens: [lemmatizer.lemmatize(t) for t in tokens]
)

# Comparison
stem_unique = set(t for tokens in df['stemmed'] for t in tokens)
lemma_unique = set(t for tokens in df['lemmatized'] for t in tokens)
print(f"Token unik setelah stemming: {len(stem_unique)}")
print(f"Token unik setelah lemmatisation: {len(lemma_unique)}")
print(f"\nContoh stemming vs lemmatisation:")
for word in ['detected', 'flooding', 'connections', 'suspicious']:
    print(f"  {word:15s} → stem: {stemmer.stem(word):12s}, lemma: {lemmatizer.lemmatize(word)}")


---
## Soal 5: Analysing Ngrams

Temukan bigram dan trigram paling umum dari semua log messages. Ini membantu mengidentifikasi pola kata yang sering muncul bersama.


In [ ]:

from nltk.util import ngrams

# Prepare text: lowercase, no stop words
all_filtered = [t for tokens in df['filtered_tokens'] for t in tokens]

# Bigrams
bigrams = list(ngrams(all_filtered, 2))
top_bigrams = Counter(bigrams).most_common(10)
print("=== Top 10 Bigrams ===")
for bg, count in top_bigrams:
    print(f"  {' '.join(bg)}: {count}")

# Trigrams
trigrams = list(ngrams(all_filtered, 3))
top_trigrams = Counter(trigrams).most_common(10)
print("\n=== Top 10 Trigrams ===")
for tg, count in top_trigrams:
    print(f"  {' '.join(tg)}: {count}")


---
## Soal 6: Creating Word Clouds

Buat word cloud dari:
1. Malicious traffic log messages
2. Benign traffic log messages
Stop words sudah difilter. Warna berbeda untuk masing-masing.


In [ ]:

from wordcloud import WordCloud
import matplotlib.pyplot as plt

# CVE-equivalent: Malicious traffic words
malicious_text = ' '.join(
    df[df['traffic_type'] == 'Malicious']['log_message'].values
)
benign_text = ' '.join(
    df[df['traffic_type'] == 'Benign']['log_message'].values
)

try:
    wc_mal = WordCloud(width=600, height=300, background_color='white',
                       stopwords=stop_words, colormap='Reds').generate(malicious_text)
    wc_ben = WordCloud(width=600, height=300, background_color='white',
                       stopwords=stop_words, colormap='Greens').generate(benign_text)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(wc_mal, interpolation='bilinear')
    axes[0].set_title('Word Cloud — Malicious Traffic')
    axes[0].axis('off')
    axes[1].imshow(wc_ben, interpolation='bilinear')
    axes[1].set_title('Word Cloud — Benign Traffic')
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('/tmp/ch8_wordcloud.png', dpi=150)
    print("✅ Word cloud saved to /tmp/ch8_wordcloud.png")
except ImportError:
    # Fallback: bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for i, (tt, color) in enumerate([('Malicious', 'red'), ('Benign', 'green')]):
        tokens = [t for row in df[df['traffic_type'] == tt]['filtered_tokens'] for t in row]
        top = Counter(tokens).most_common(10)
        words, counts = zip(*top)
        axes[i].barh(words, counts, color=color, alpha=0.7)
        axes[i].set_title(f'Top Words — {tt}')
    plt.savefig('/tmp/ch8_wordcloud_fallback.png', dpi=150)
    print("✅ Fallback bar chart saved")


---
## Soal 7: Checking Term Frequency (TF-IDF)

Hitung TF-IDF untuk semua log messages. Temukan:
1. Top 15 terms dengan TF-IDF tertinggi
2. Bandingkan TF-IDF antara Malicious vs Benign


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

texts = [' '.join(tokens) for tokens in df['filtered_tokens']]
vectorizer = TfidfVectorizer(max_features=50, stop_words=list(stop_words))
tfidf_matrix = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

# Top TF-IDF terms
tfidf_sums = tfidf_matrix.sum(axis=0).A1
top_tfidf = sorted(zip(feature_names, tfidf_sums), key=lambda x: x[1], reverse=True)[:15]
print("=== Top 15 TF-IDF Terms ===")
for term, score in top_tfidf:
    print(f"  {term}: {score:.4f}")

# Compare Malicious vs Benign
mal_idx = df[df['traffic_type'] == 'Malicious'].index
ben_idx = df[df['traffic_type'] == 'Benign'].index
mal_tfidf = tfidf_matrix[mal_idx].sum(axis=0).A1
ben_tfidf = tfidf_matrix[ben_idx].sum(axis=0).A1

print("\n=== TF-IDF: Malicious vs Benign (top 5) ===")
for i, (name, m, b) in enumerate(sorted(zip(feature_names, mal_tfidf, ben_tfidf), key=lambda x: abs(x[1]-x[2]), reverse=True)[:5]):
    print(f"  {name:15s} → Mal: {m:.4f}, Ben: {b:.4f}")


---
## Soal 8: Checking Sentiments

Analisis sentiment dari log messages. Apakah alert Malicious memiliki sentiment lebih negatif? Bandingkan sentiment score per traffic_type.


In [ ]:

from textblob import TextBlob

df['sentiment'] = df['log_message'].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

print("=== Sentiment per Traffic Type ===")
sent_by_type = df.groupby('traffic_type')['sentiment'].agg(['mean', 'std', 'min', 'max'])
print(sent_by_type.round(4))

plt.figure(figsize=(6, 4))
df.boxplot(column='sentiment', by='traffic_type')
plt.title('Sentiment Score per Traffic Type')
plt.suptitle('')
plt.tight_layout()
plt.savefig('/tmp/ch8_sentiment.png', dpi=150)
print("✅ Sentiment chart saved")


---
## Soal 9: Performing Topic Modelling (LDA)

Gunakan **LDA (Latent Dirichlet Allocation)** untuk menemukan topik tersembunyi dalam log messages. Set 4 topics. Interpretasikan setiap topik.


In [ ]:

import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel

# Prepare corpus
texts_lem = df['lemmatized'].tolist()
dictionary = corpora.Dictionary(texts_lem)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in texts_lem]

# Train LDA (4 topics)
lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=4, random_state=42, passes=10)

print("=== LDA Topics (4 topics) ===")
for topic_id, topic in lda.print_topics(num_words=8):
    print(f"\nTopic {topic_id}: {topic}")

# Assign dominant topic
def get_dominant_topic(bow):
    topics = lda.get_document_topics(bow)
    return max(topics, key=lambda x: x[1])[0]

df['dominant_topic'] = corpus_to_bow = [dictionary.doc2bow(t) for t in texts_lem]
df['dominant_topic'] = [get_dominant_topic(bow) for bow in corpus]

print("\n=== Topic Distribution ===")
print(df['dominant_topic'].value_counts())


---
## Soal 10: Choosing Optimal Number of Topics

Cari jumlah topic optimal dengan menghitung **Coherence Score** untuk num_topics = 2 hingga 8. Pilih jumlah topic dengan coherence tertinggi.


In [ ]:

from gensim.models import CoherenceModel

coherence_scores = []
topic_range = range(2, 9)

for num_topics in topic_range:
    lda_temp = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics,
                        random_state=42, passes=10)
    coherence = CoherenceModel(model=lda_temp, texts=texts_lem,
                                dictionary=dictionary, coherence='c_v').get_coherence()
    coherence_scores.append(coherence)
    print(f"  Topics={num_topics}: Coherence={coherence:.4f}")

best_topics = list(topic_range)[coherence_scores.index(max(coherence_scores))]
print(f"\n✅ Optimal topics: {best_topics} (score: {max(coherence_scores):.4f})")

plt.figure(figsize=(8, 4))
plt.plot(topic_range, coherence_scores, 'bo-')
plt.axvline(best_topics, color='r', linestyle='--', label=f'Optimal: {best_topics}')
plt.xlabel('Number of Topics')
plt.ylabel('Coherence Score')
plt.title('Coherence Score vs Number of Topics')
plt.legend()
plt.tight_layout()
plt.savefig('/tmp/ch8_coherence.png', dpi=150)
print("✅ Coherence chart saved")
